In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Integrated feature-gene selection and plotting utilities (single-file).
Functions:
  - find_marker_genes: per-cluster Mann-Whitney test + BH correction -> CSV
  - plot_gene_on_umap: plot current / delta / future expression on UMAP
  - plot_spatial_heatmap: grid-smoothed spatial heatmap for a gene
  - transcript_distance_summary: per-cell transcript counts and median distance-to-nuclear-boundary
  - several cell-type visualization helpers (UMAP/spatial/combined/arrows)
Edit CONFIG block before running.
"""

import os
import math
import random
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# ---------------- CONFIG ----------------
NORM_MODE = os.environ.get("SPATIAL_VELO_NORM_MODE", "2D").upper()
_CWD_DATA_DIR = os.path.join(os.getcwd(), "data", "test_data")
_DEFAULT_WORKDIR = (
    _CWD_DATA_DIR
    if os.path.isdir(_CWD_DATA_DIR)
    else os.path.join("..", "data", "test_data")
)
BASE_DIR = os.path.abspath(
    os.environ.get("SPATIAL_VELO_WORKDIR", _DEFAULT_WORKDIR)
)
DELTA_FILE = os.path.join(BASE_DIR, "pipeline_out", f"mode_{NORM_MODE}", "out_part3", "delta_export_prob_median.csv") # delta matrix from previous steps
META_FILE = os.path.join(BASE_DIR, "xenium_test_metadata.csv")  # metadata file path

# optional, a seperate file for cell type annonation(contain cell/SCINA_label column)
# if CLUSTER_FILE not exist, fall back to find cell type info in metadata
CLUSTER_FILE = os.path.join(BASE_DIR, "R", "cell_clusters_SCINA.csv")  

CURRENT_EXPR_FILE = os.path.join(BASE_DIR, "pipeline_out", f"mode_{NORM_MODE}", "out_part3", "transcripts_cytoplasm_matrix_common.csv")  # cytoplasmic expression file from Step 1
UMAP_WITH_META_FILE = os.path.join(BASE_DIR, "pipeline_out", f"mode_{NORM_MODE}", "out_part3", "umap_with_meta.csv")  #file from previous steps
TX_NUC_FILE = os.path.join(BASE_DIR, "pipeline_out", f"mode_{NORM_MODE}", "out_part1", "tx_nuclear_with_rnorm.csv")   #file from previous steps
OUT_DIR = os.path.join(BASE_DIR, "pipeline_out", f"mode_{NORM_MODE}", "out_part3", "feature_genes")  #output file path
os.makedirs(OUT_DIR, exist_ok=True)

# Analysis parameters
TOPK = 10                     #number of top-ranked marker genes reported per cluster
MIN_CELLS = 10                #exclude clusters with less than certain number of cells
GRID_N_SPATIAL = 250          # spatial heatmap resolution (reduce for speed if slow)
GAUSSIAN_SIGMA_SPATIAL = 1.2  #Gaussian smoothing strength in the spatial delta field, typically 1(noisy)-2.5(smoothed)
UMAP_POINTS_SIZE = 32         #dot size

# UMAP plotting for an example gene
example_gene = "CD19"  # change as desired

# ---------------- Utilities ----------------
def bh_adjust(pvals):
    """Benjamini-Hochberg FDR correction (returns adjusted p-values)."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = np.empty(n, dtype=float)
    ranked[order] = np.arange(1, n+1)
    adj = p * n / ranked
    adj_sorted = adj[order]
    adj_cummin = np.minimum.accumulate(adj_sorted[::-1])[::-1]
    adj_corrected = np.empty(n, dtype=float)
    adj_corrected[order] = adj_cummin
    return np.minimum(adj_corrected, 1.0)

def read_table_indexed(path, index_col=0):
    """Read CSV with index_col default 0 and ensure index is string."""
    df = pd.read_csv(path, index_col=index_col)
    # ensure string index
    df.index = df.index.astype(str)
    return df

def safe_read_csv(path):
    """Read CSV without forcing an index (used when structure unknown)."""
    return pd.read_csv(path)

def guess_cell_col(df):
    """Guess the cell id column name in dataframe (index-based or common column names)."""
    if df.index.name and df.index.name != 'Unnamed: 0':
        return None  # index used
    for c in ["cell", "cell_id", "Unnamed: 0", "barcode"]:
        if c in df.columns:
            return c
    return None

# ---------------- Helper: get cell-type info ----------------
def get_cell_type_info(meta_file, cluster_file=None):
    """
    Load metadata and optional cluster file, return merged_meta and meta_cell_col.
    merged_meta contains 'cluster_label' column.
    """
    meta = pd.read_csv(meta_file)
    meta_cell_col = "Unnamed: 0" if "Unnamed: 0" in meta.columns else ("cell" if "cell" in meta.columns else meta.columns[0])
    meta[meta_cell_col] = meta[meta_cell_col].astype(str)

    if cluster_file and os.path.exists(cluster_file):
        cluster_df = pd.read_csv(cluster_file)
        cluster_cell_col = next((c for c in ["cell","cell_id","Unnamed: 0"] if c in cluster_df.columns),
                                cluster_df.columns[0])
        cluster_df[cluster_cell_col] = cluster_df[cluster_cell_col].astype(str)
        merged_meta = pd.merge(meta, cluster_df, left_on=meta_cell_col, right_on=cluster_cell_col, how="inner")
        # find cluster label column
        for col in ['SCINA_label', 'SCINA_label_mod', 'cluster', 'label', 'cell_type', 'celltype']:
            if col in merged_meta.columns:
                merged_meta['cluster_label'] = merged_meta[col].astype(str)
                break
        if 'cluster_label' not in merged_meta.columns:
            possible = [c for c in merged_meta.columns if 'cluster' in c.lower() or 'label' in c.lower() or 'type' in c.lower()]
            if possible:
                merged_meta['cluster_label'] = merged_meta[possible[0]].astype(str)
            else:
                raise RuntimeError("Cannot find cluster label column in merged metadata.")
    else:
        # try to extract cell type from metadata itself
        merged_meta = meta.copy()
        for col in ['cell_type', 'celltype', 'CellType', 'Cell_type']:
            if col in merged_meta.columns:
                merged_meta['cluster_label'] = merged_meta[col].astype(str)
                break
        if 'cluster_label' not in merged_meta.columns:
            possible = [c for c in merged_meta.columns if 'type' in c.lower() or 'cluster' in c.lower() or 'label' in c.lower()]
            if possible:
                merged_meta['cluster_label'] = merged_meta[possible[0]].astype(str)
            else:
                raise RuntimeError("Cannot find cell type / cluster column in metadata.")
    return merged_meta, meta_cell_col

# ---------------- 1) Marker gene computation ----------------
def find_marker_genes(delta_file, meta_file, cluster_file, out_dir,
                      topk=TOPK, min_cells=MIN_CELLS):
    """Compute per-cluster marker genes from delta matrix (cell x gene)."""
    print(">> find_marker_genes: loading files...")
    if not os.path.exists(delta_file):
        raise FileNotFoundError(f"delta file not found: {delta_file}")
    if not os.path.exists(meta_file):
        raise FileNotFoundError(f"metadata file not found: {meta_file}")

    merged_meta, meta_cell_col = None, None
    # get cell-type info
    merged_meta, meta_cell_col = get_cell_type_info(meta_file, cluster_file)

    delta = pd.read_csv(delta_file, index_col=0)
    delta.index = delta.index.astype(str)
    common_cells = merged_meta[meta_cell_col].astype(str).values
    delta_sub = delta.loc[delta.index.intersection(common_cells)].copy()

    if delta_sub.shape[0] == 0:
        raise RuntimeError("No overlapping cells between delta matrix and metadata/cluster file.")

    meta_aligned = merged_meta.set_index(meta_cell_col).loc[delta_sub.index].copy()
    meta_aligned['cluster_label'] = meta_aligned['cluster_label'].astype(str)

    genes = delta_sub.columns.tolist()
    clusters = sorted(meta_aligned['cluster_label'].unique().tolist())
    print(f"  Cells: {delta_sub.shape[0]}, Genes: {len(genes)}, Clusters: {len(clusters)}")

    all_results = []
    for cl in clusters:
        mask_in = (meta_aligned['cluster_label'] == cl).values
        mask_out = ~mask_in
        n_in = mask_in.sum()
        n_out = mask_out.sum()
        if n_in <= min_cells:
            print(f"  Skip cluster {cl}: n_in={n_in} <= {min_cells}")
            continue
        mean_in = delta_sub.loc[mask_in].mean(axis=0)
        mean_out = delta_sub.loc[mask_out].mean(axis=0)
        log2fc = np.log2((mean_in + 1) / (mean_out + 1))
        pvals = []
        for g in genes:
            x = delta_sub.loc[mask_in, g].values
            y = delta_sub.loc[mask_out, g].values
            if np.all(x == 0) and np.all(y == 0):
                pvals.append(1.0)
            else:
                try:
                    _, p = mannwhitneyu(x, y, alternative='two-sided')
                    pvals.append(p if not math.isnan(p) else 1.0)
                except Exception:
                    pvals.append(1.0)
        p_adj = bh_adjust(pvals)
        dfc = pd.DataFrame({
            'gene': genes,
            'mean_in': mean_in.values,
            'mean_out': mean_out.values,
            'log2FC': log2fc.values,
            'pvalue': pvals,
            'p_adj': p_adj
        })
        dfc['cluster'] = cl
        dfc_sorted = dfc.sort_values(by=['log2FC', 'p_adj'], ascending=[False, True])
        top_genes = dfc_sorted.head(topk)
        print(f"\nCluster {cl} (n_in={n_in}, n_out={n_out}) top {topk} genes:")
        print(top_genes[['gene','mean_in','mean_out','log2FC','p_adj']].to_string(index=False))
        all_results.append(dfc_sorted)

    if len(all_results) > 0:
        combined = pd.concat(all_results, axis=0)
        out_csv = os.path.join(out_dir, "marker_genes_all_clusters_full.csv")
        combined.to_csv(out_csv, index=False)
        print(f"\nSaved combined marker gene table: {out_csv}")
    else:
        print("No clusters passed the MIN_CELLS threshold; no results saved.")
    return

# ---------------- 2) UMAP gene plotting ----------------
def plot_gene_on_umap(gene, out_dir,
                      umap_file=UMAP_WITH_META_FILE,
                      current_expr_file=CURRENT_EXPR_FILE,
                      delta_file=DELTA_FILE):
    """
    Plot gene expression on UMAP for current, delta, future (current+delta).
    """
    print(f">> plot_gene_on_umap: {gene}")
    if not os.path.exists(umap_file):
        raise FileNotFoundError(f"UMAP file not found: {umap_file}")
    umap_df = pd.read_csv(umap_file, index_col=0)
    umap_df.index = umap_df.index.astype(str)

    if not os.path.exists(delta_file):
        raise FileNotFoundError(f"Delta file not found: {delta_file}")
    delta_mat = pd.read_csv(delta_file, index_col=0)
    delta_mat.index = delta_mat.index.astype(str)

    # find delta expr for gene (case-insensitive)
    cols_lower = {c.lower(): c for c in delta_mat.columns}
    if gene in delta_mat.columns:
        delta_expr = delta_mat[gene].reindex(umap_df.index).fillna(0.0)
    elif gene.lower() in cols_lower:
        delta_expr = delta_mat[cols_lower[gene.lower()]].reindex(umap_df.index).fillna(0.0)
    else:
        delta_expr = pd.Series(0.0, index=umap_df.index)

    # current expr
    if os.path.exists(current_expr_file):
        current = pd.read_csv(current_expr_file, index_col=0)
        current.index = current.index.astype(str)
        cols_lower_cur = {c.lower(): c for c in current.columns}
        if gene in current.columns:
            current_expr = current[gene].reindex(umap_df.index).fillna(0.0)
        elif gene.lower() in cols_lower_cur:
            current_expr = current[cols_lower_cur[gene.lower()]].reindex(umap_df.index).fillna(0.0)
        else:
            print(f"  Warning: gene {gene} not found in current expression; using zeros.")
            current_expr = pd.Series(0.0, index=umap_df.index)
    else:
        print(f"  Current expression file not found ({current_expr_file}); using zeros.")
        current_expr = pd.Series(0.0, index=umap_df.index)

    future_expr = current_expr + delta_expr

    gray_to_green = LinearSegmentedColormap.from_list("gray_green", ["#d3d3d3", "#006400"])
    datasets = {
        "current": (current_expr, gray_to_green),
        "delta": (delta_expr, "coolwarm"),
        "future": (future_expr, "YlOrRd")
    }

    X = umap_df[["UMAP1","UMAP2"]].values
    x_min, x_max = X[:,0].min(), X[:,0].max()
    y_min, y_max = X[:,1].min(), X[:,1].max()

    for name, (expr, cmap) in datasets.items():
        fig, ax = plt.subplots(figsize=(8,8))
        expr_vals = expr.values
        if name == "current":
            if expr_vals.max() - expr_vals.min() > 1e-9:
                expr_norm = (expr_vals - expr_vals.min()) / (expr_vals.max() - expr_vals.min() + 1e-9)
                alphas = 0.1 + 0.7 * expr_norm
            else:
                alphas = 0.5
        else:
            alphas = 0.6
        sc = ax.scatter(X[:,0], X[:,1],
                        c=expr_vals, cmap=cmap, s=UMAP_POINTS_SIZE, edgecolors="none",
                        alpha=alphas)
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
        ax.set_title(f"{gene} - {name}")
        plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.04)
        plt.tight_layout()
        out_png = os.path.join(out_dir, f"{gene}_{name}_umap.png")
        plt.savefig(out_png, dpi=300)
        plt.close()
        print(f"  Saved: {out_png}")

# ---------------- 3) Spatial heatmap for single gene ----------------
def plot_spatial_heatmap(gene, expr_file, meta_file, out_dir,
                         grid_n=GRID_N_SPATIAL, sigma=GAUSSIAN_SIGMA_SPATIAL,
                         transform=None, cmap="viridis", show_points=False):
    """Plot a grid-smoothed heatmap for `gene` on spatial coordinates from `meta_file`."""
    print(f">> plot_spatial_heatmap: {gene}")
    if not os.path.exists(expr_file):
        raise FileNotFoundError(f"Expression file not found: {expr_file}")
    if not os.path.exists(meta_file):
        raise FileNotFoundError(f"Metadata file not found: {meta_file}")

    expr = pd.read_csv(expr_file, index_col=0)
    expr.index = expr.index.astype(str)
    meta = pd.read_csv(meta_file, index_col=0)
    meta.index = meta.index.astype(str)

    # detect coordinate columns (try a few common names)
    X_col = next((c for c in meta.columns if ('center' in c.lower() and 'x' in c.lower()) or ('global' in c.lower() and 'x' in c.lower()) or c.lower() in ['x','pos_x','posx']), None)
    Y_col = next((c for c in meta.columns if ('center' in c.lower() and 'y' in c.lower()) or ('global' in c.lower() and 'y' in c.lower()) or c.lower() in ['y','pos_y','posy']), None)
    if X_col is None or Y_col is None:
        # fallback
        if 'center_x' in meta.columns and 'center_y' in meta.columns:
            X_col, Y_col = 'center_x', 'center_y'
        else:
            raise RuntimeError("Cannot find spatial coordinate columns in metadata (look for center_x / center_y / x / y / global_x).")

    cols_lower = {c.lower(): c for c in expr.columns}
    if gene in expr.columns:
        gene_col = gene
    elif gene.lower() in cols_lower:
        gene_col = cols_lower[gene.lower()]
    else:
        raise RuntimeError(f"Gene {gene} not found in expression file.")

    common = expr.index.intersection(meta.index)
    if len(common) == 0:
        raise RuntimeError("No shared cell ids between expression and metadata.")

    expr_sub = expr.loc[common, gene_col].astype(float).values
    meta_sub = meta.loc[common]
    X = meta_sub[X_col].astype(float).values
    Y = meta_sub[Y_col].astype(float).values

    if transform == "log1p":
        vals_plot = np.log1p(expr_sub)
    elif transform == "sqrt":
        vals_plot = np.sqrt(expr_sub)
    else:
        vals_plot = expr_sub

    # grid averaging
    x_edges = np.linspace(X.min(), X.max(), grid_n + 1)
    y_edges = np.linspace(Y.min(), Y.max(), grid_n + 1)
    grid = np.zeros((grid_n, grid_n), dtype=float)
    counts = np.zeros_like(grid, dtype=int)
    xi = np.searchsorted(x_edges, X, side='right') - 1
    yi = np.searchsorted(y_edges, Y, side='right') - 1
    xi = np.clip(xi, 0, grid_n-1)
    yi = np.clip(yi, 0, grid_n-1)
    for a, b, v in zip(xi, yi, vals_plot):
        grid[b, a] += v
        counts[b, a] += 1
    mask = counts > 0
    grid[mask] = grid[mask] / counts[mask]
    grid_smooth = gaussian_filter(grid, sigma=sigma)

    extent = (X.min(), X.max(), Y.min(), Y.max())
    plt.figure(figsize=(8,8))
    im = plt.imshow(grid_smooth, origin='lower', extent=extent, cmap=cmap, aspect='equal')
    plt.title(f"Spatial heatmap — {gene} (transform={transform})")
    plt.xlabel(X_col); plt.ylabel(Y_col)
    cbar = plt.colorbar(im)
    cbar.set_label(f"{gene} ({transform if transform else 'raw'})")
    if show_points:
        plt.scatter(X, Y, c='k', s=4, alpha=0.6, linewidths=0)
    out_png = os.path.join(out_dir, f"{gene}_spatial_heatmap.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()
    print(f"  Saved spatial heatmap: {out_png}")

# ---------------- 4) Transcript-distance summary ----------------
def transcript_distance_summary(tx_file, meta_file, gene, out_dir):
    """
    Compute per-cell transcript counts and median distance-to-boundary for a gene,
    then save per-cell-type summary (mean ± SEM and SD).
    """
    print(f">> transcript_distance_summary: {gene}")
    if not os.path.exists(tx_file):
        raise FileNotFoundError(f"tx file not found: {tx_file}")
    if not os.path.exists(meta_file):
        raise FileNotFoundError(f"meta file not found: {meta_file}")

    tx_df = pd.read_csv(tx_file)
    meta_df = pd.read_csv(meta_file)

    if 'r_norm' not in tx_df.columns:
        raise RuntimeError("tx file must contain 'r_norm' column.")
    tx_df['dist_to_boundary'] = 1.0 - tx_df['r_norm']

    gene_col_name = 'target' if 'target' in tx_df.columns else ('gene' if 'gene' in tx_df.columns else None)
    if gene_col_name is None:
        raise RuntimeError("Cannot find 'target' or 'gene' column in tx file.")

    gene_tx = tx_df[tx_df[gene_col_name] == gene].copy()
    if gene_tx.empty:
        print(f"No transcripts found for gene {gene}.")
        return None

    cell_stats = gene_tx.groupby('cell').agg(
        tx_count=(gene_col_name, 'count'),
        median_dist=('dist_to_boundary', 'median')
    ).reset_index()

    if 'cell' in meta_df.columns:
        merged = cell_stats.merge(meta_df[['cell','cell_type']].drop_duplicates(), on='cell', how='left')
    elif 'Unnamed: 0' in meta_df.columns:
        merged = cell_stats.merge(meta_df[['Unnamed: 0','cell_type']].drop_duplicates(), left_on='cell', right_on='Unnamed: 0', how='left')
    else:
        merged = cell_stats.copy()
        merged['cell_type'] = np.nan

    summary_sem = merged.groupby('cell_type', dropna=True).agg(
        mean_tx=('tx_count','mean'),
        sem_tx=('tx_count', lambda x: x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else np.nan),
        mean_dist=('median_dist','mean'),
        sem_dist=('median_dist', lambda x: x.std(ddof=1)/np.sqrt(len(x)) if len(x)>1 else np.nan)
    ).reset_index()

    summary_sd = merged.groupby('cell_type', dropna=True).agg(
        mean_tx=('tx_count','mean'),
        sd_tx=('tx_count','std'),
        mean_dist=('median_dist','mean'),
        sd_dist=('median_dist','std')
    ).reset_index()

    out1 = os.path.join(out_dir, f"{gene}_tx_stats_sem.csv")
    out2 = os.path.join(out_dir, f"{gene}_tx_stats_sd.csv")
    summary_sem.to_csv(out1, index=False)
    summary_sd.to_csv(out2, index=False)
    print(f"  Saved tx summaries: {out1}, {out2}")
    return merged, summary_sem, summary_sd

# ---------------- 5) Cell-type plotting helpers ----------------
def plot_celltype_distribution(umap_file, meta_file, cluster_file, out_dir):
    """Plot cell type distribution on UMAP and spatial coordinates."""
    print(">> plot_celltype_distribution")
    merged_meta, meta_cell_col = get_cell_type_info(meta_file, cluster_file)
    if not os.path.exists(umap_file):
        raise FileNotFoundError(f"UMAP file not found: {umap_file}")
    umap_df = pd.read_csv(umap_file)
    if 'cell' not in umap_df.columns and 'cell_id' in umap_df.columns:
        umap_df = umap_df.rename(columns={'cell_id': 'cell'})
    celltype_umap = pd.merge(umap_df, merged_meta[[meta_cell_col, 'cluster_label']],
                             left_on='cell', right_on=meta_cell_col, how='inner')

    cell_types = celltype_umap['cluster_label'].unique()
    n_types = len(cell_types)
    colors = plt.cm.tab20(np.linspace(0, 1, n_types))
    color_map = {cell_type: colors[i] for i, cell_type in enumerate(cell_types)}
    legend_elements = [mpatches.Patch(color=color_map[cell_type], label=cell_type) for cell_type in cell_types]

    plt.figure(figsize=(10, 8))
    for cell_type in cell_types:
        mask = celltype_umap['cluster_label'] == cell_type
        plt.scatter(celltype_umap.loc[mask, 'UMAP1'], celltype_umap.loc[mask, 'UMAP2'],
                    c=[color_map[cell_type]], s=8, alpha=0.7, label=cell_type)
    plt.xlabel("UMAP1"); plt.ylabel("UMAP2"); plt.title("Cell Type Distribution on UMAP")
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    umap_out = os.path.join(out_dir, "celltype_umap_distribution.png")
    plt.savefig(umap_out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved UMAP cell type distribution: {umap_out}")

    # spatial plot: try to detect coordinates in merged_meta
    X_col = next((c for c in merged_meta.columns if 'center' in c.lower() and 'x' in c.lower()), None)
    Y_col = next((c for c in merged_meta.columns if 'center' in c.lower() and 'y' in c.lower()), None)
    # fallback to 'global' names
    if X_col is None or Y_col is None:
        X_col = next((c for c in merged_meta.columns if 'global' in c.lower() and 'x' in c.lower()), X_col)
        Y_col = next((c for c in merged_meta.columns if 'global' in c.lower() and 'y' in c.lower()), Y_col)
    if X_col is None or Y_col is None:
        print("  Cannot find spatial coordinate columns, skipping spatial distribution plot.")
        return

    plt.figure(figsize=(12, 10))
    for cell_type in cell_types:
        mask = merged_meta['cluster_label'] == cell_type
        plt.scatter(merged_meta.loc[mask, X_col], merged_meta.loc[mask, Y_col],
                    c=[color_map[cell_type]], s=8, alpha=0.7, label=cell_type)
    plt.xlabel(X_col); plt.ylabel(Y_col); plt.title("Cell Type Distribution on Spatial Coordinates")
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    spatial_out = os.path.join(out_dir, "celltype_spatial_distribution.png")
    plt.savefig(spatial_out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved spatial cell type distribution: {spatial_out}")

def plot_celltype_with_arrows(umap_file, meta_file, cluster_file, out_dir):
    """Plot cell type distribution on UMAP and spatial coordinates (placeholder for arrows)."""
    print(">> plot_celltype_with_arrows")
    merged_meta, meta_cell_col = get_cell_type_info(meta_file, cluster_file)
    if not os.path.exists(umap_file):
        raise FileNotFoundError(f"UMAP file not found: {umap_file}")
    umap_df = pd.read_csv(umap_file)
    if 'cell' not in umap_df.columns and 'cell_id' in umap_df.columns:
        umap_df = umap_df.rename(columns={'cell_id': 'cell'})
    celltype_umap = pd.merge(umap_df, merged_meta[[meta_cell_col, 'cluster_label']],
                             left_on='cell', right_on=meta_cell_col, how='inner')

    cell_types = celltype_umap['cluster_label'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(cell_types)))
    color_map = {cell_type: colors[i] for i, cell_type in enumerate(cell_types)}
    legend_elements = [mpatches.Patch(color=color_map[cell_type], label=cell_type) for cell_type in cell_types]

    plt.figure(figsize=(10,8))
    for ct in cell_types:
        mask = celltype_umap['cluster_label'] == ct
        plt.scatter(celltype_umap.loc[mask, 'UMAP1'], celltype_umap.loc[mask, 'UMAP2'],
                    c=[color_map[ct]], s=8, alpha=0.7, label=ct)
    plt.xlabel("UMAP1"); plt.ylabel("UMAP2"); plt.title("Cell Type Distribution on UMAP with Arrows")
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    umap_arrow_out = os.path.join(out_dir, "celltype_umap_with_arrows.png")
    plt.savefig(umap_arrow_out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved UMAP cell type distribution with arrows: {umap_arrow_out}")

    # spatial (no real arrows; placeholder)
    X_col = next((c for c in merged_meta.columns if 'center' in c.lower() and 'x' in c.lower()), None)
    Y_col = next((c for c in merged_meta.columns if 'center' in c.lower() and 'y' in c.lower()), None)
    if X_col is None or Y_col is None:
        print("  Cannot find spatial coordinate columns, skipping spatial with arrows.")
        return
    plt.figure(figsize=(12,10))
    for ct in cell_types:
        mask = merged_meta['cluster_label'] == ct
        plt.scatter(merged_meta.loc[mask, X_col], merged_meta.loc[mask, Y_col],
                    c=[color_map[ct]], s=8, alpha=0.7, label=ct)
    plt.xlabel(X_col); plt.ylabel(Y_col); plt.title("Cell Type Distribution on Spatial Coordinates with Arrows")
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    spatial_arrow_out = os.path.join(out_dir, "celltype_spatial_with_arrows.png")
    plt.savefig(spatial_arrow_out, dpi=300, bbox_inches='tight'); plt.close()
    print(f"  Saved spatial cell type distribution with arrows: {spatial_arrow_out}")

def plot_celltype_space_and_umap(meta_file, umap_file, out_dir=OUT_DIR):
    """Plot combined figure: left = spatial coordinates, right = UMAP (colored by cell_type)."""
    print(">> plot_celltype_space_and_umap")
    meta = pd.read_csv(meta_file)
    umap = pd.read_csv(umap_file)
    # ensure 'cell' present in both
    if 'cell' not in meta.columns and 'Unnamed: 0' in meta.columns:
        meta = meta.rename(columns={'Unnamed: 0': 'cell'})
    if 'cell' not in umap.columns and 'cell_id' in umap.columns:
        umap = umap.rename(columns={'cell_id': 'cell'})
    df = pd.merge(meta, umap, on="cell", how="inner")

    if 'cell_type' not in df.columns:
        raise RuntimeError("plot_celltype_space_and_umap: 'cell_type' column not found in merged data.")

    cell_types = df["cell_type"].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(cell_types)))
    color_map = dict(zip(cell_types, colors))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ct in cell_types:
        subset = df[df["cell_type"] == ct]
        axes[0].scatter(subset["x"], subset["y"], s=5, color=color_map[ct], label=ct, alpha=0.7)
    axes[0].set_title("Cell types in spatial coordinates")
    axes[0].set_xlabel("X"); axes[0].set_ylabel("Y"); axes[0].invert_yaxis()

    for ct in cell_types:
        subset = df[df["cell_type"] == ct]
        axes[1].scatter(subset["UMAP1"], subset["UMAP2"], s=5, color=color_map[ct], label=ct, alpha=0.7)
    axes[1].set_title("Cell types in UMAP"); axes[1].set_xlabel("UMAP1"); axes[1].set_ylabel("UMAP2")

    handles = [mpatches.Patch(color=color_map[ct], label=ct) for ct in cell_types]
    fig.legend(handles=handles, loc="center right", bbox_to_anchor=(1.15, 0.5))
    plt.tight_layout()
    save_path = os.path.join(out_dir, "celltype_space_umap.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.close()
    print(f"  Saved {save_path}")
def plot_celltype_space_umap_with_arrows(meta_file, umap_file, out_dir=OUT_DIR, n_arrows=500, arrow_scale=None):
    """Plot UMAP and draw arrows from spatial coords to UMAP coords (sampled)."""
    print(">> plot_celltype_space_umap_with_arrows")
    meta = pd.read_csv(meta_file)
    umap = pd.read_csv(umap_file)

    # 确保 cell 列一致
    if 'cell' not in meta.columns and 'Unnamed: 0' in meta.columns:
        meta = meta.rename(columns={'Unnamed: 0':'cell'})
    if 'cell' not in umap.columns and 'cell_id' in umap.columns:
        umap = umap.rename(columns={'cell_id':'cell'})

    df = pd.merge(meta, umap, on="cell", how="inner")

    # 尝试识别 cell type 列
    if 'cell_type' not in df.columns:
        for col in ['cluster_label', 'celltype', 'CellType', 'Cell_type']:
            if col in df.columns:
                df['cell_type'] = df[col].astype(str)
                break
    if 'cell_type' not in df.columns:
        raise RuntimeError("plot_celltype_space_umap_with_arrows: 'cell_type' column not found in merged data.")

    cell_types = df["cell_type"].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(cell_types)))
    color_map = dict(zip(cell_types, colors))

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    for ct in cell_types:
        subset = df[df["cell_type"] == ct]
        ax.scatter(subset["UMAP1"], subset["UMAP2"], s=5, color=color_map[ct], label=ct, alpha=0.7)

    df_sample = df.sample(min(n_arrows, len(df)), random_state=0)

    # 自动计算箭头缩放比例
    if arrow_scale is None:
        delta = df_sample[["UMAP1","UMAP2"]].values - df_sample[["x","y"]].values
        mean_delta = np.mean(np.linalg.norm(delta, axis=1))
        arrow_scale = 0.3 / mean_delta  # 调整到合理可见的长度

    for _, row in df_sample.iterrows():
        dx = (row["UMAP1"] - row["x"]) * arrow_scale
        dy = (row["UMAP2"] - row["y"]) * arrow_scale
        ax.arrow(row["x"], row["y"], dx, dy,
                 head_width=0.3*arrow_scale, head_length=0.3*arrow_scale,
                 fc="k", ec="k", alpha=0.3, length_includes_head=True)

    ax.set_title("Cell types in UMAP with arrows from spatial")
    ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
    handles = [mpatches.Patch(color=color_map[ct], label=ct) for ct in cell_types]
    ax.legend(handles=handles, loc="best", fontsize=8)

    save_path = os.path.join(out_dir, "celltype_umap_with_arrows.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.close()
    print(f"  Saved {save_path}")
# ---------------- Main ----------------
if __name__ == "__main__":
    print("=== Feature gene utilities (single-file) starting ===")
    # 1) Marker genes
    try:
        cluster_file_to_use = CLUSTER_FILE if os.path.exists(CLUSTER_FILE) else None
        find_marker_genes(DELTA_FILE, META_FILE, cluster_file_to_use, OUT_DIR,
                          topk=TOPK, min_cells=MIN_CELLS)
    except Exception as e:
        print(f"[marker genes] skipped/failed: {e}")

    try:
        plot_gene_on_umap(example_gene, OUT_DIR,
                          umap_file=UMAP_WITH_META_FILE,
                          current_expr_file=CURRENT_EXPR_FILE,
                          delta_file=DELTA_FILE)
    except Exception as e:
        print(f"[UMAP plots] skipped/failed: {e}")

    # 3) Spatial heatmap example
    try:
        plot_spatial_heatmap(example_gene, CURRENT_EXPR_FILE, META_FILE, OUT_DIR,
                             grid_n=GRID_N_SPATIAL, sigma=GAUSSIAN_SIGMA_SPATIAL,
                             transform=None, cmap="viridis", show_points=False)
    except Exception as e:
        print(f"[spatial heatmap] skipped/failed: {e}")

    # 4) Transcript-distance summary example
    try:
        if os.path.exists(TX_NUC_FILE):
            transcript_distance_summary(TX_NUC_FILE, META_FILE, gene=example_gene, out_dir=OUT_DIR)
        else:
            print("  tx nuclear file not found; skipping transcript distance summary.")
    except Exception as e:
        print(f"[tx distance summary] skipped/failed: {e}")

    # 5) Cell type visualizations
    try:
        plot_celltype_distribution(UMAP_WITH_META_FILE, META_FILE, CLUSTER_FILE, OUT_DIR)
    except Exception as e:
        print(f"[celltype distribution] skipped/failed: {e}")

    try:
        plot_celltype_with_arrows(UMAP_WITH_META_FILE, META_FILE, CLUSTER_FILE, OUT_DIR)
    except Exception as e:
        print(f"[celltype with arrows] skipped/failed: {e}")

    # 6) Combined spatial + UMAP figure and arrow figure (requires metadata contains 'x','y' and 'cell_type')
    try:
        plot_celltype_space_and_umap(META_FILE, UMAP_WITH_META_FILE, out_dir=OUT_DIR)
    except Exception as e:
        print(f"[space+umap] skipped/failed: {e}")
    
    try:
        arrow_scale = 0.1
        plot_celltype_space_umap_with_arrows(META_FILE, UMAP_WITH_META_FILE, out_dir=OUT_DIR,
                                             n_arrows=500, arrow_scale=arrow_scale)
    except Exception as e:
        print(f"[space->umap arrows] skipped/failed: {e}")

    print("=== Done. Check output folder:", OUT_DIR, "===")
